In [1]:
DATA_ROOT = "/kaggle/input/datasets/rudrakshjoshi5/ic-egfr-dataset/EGFR_RESEARCH_DATASET"

In [2]:

import os
import sys
import csv
import importlib.metadata as md
from pathlib import Path

import torch

# --------------------------------------------------
# PATHS
# --------------------------------------------------
DATA_ROOT = Path(
    "/kaggle/input/datasets/rudrakshjoshi5/"
    "ic-egfr-dataset/EGFR_RESEARCH_DATASET"
)

LLM_DIR = DATA_ROOT / "mutation_scientist"

print("=" * 65)
print("PYTHON ENVIRONMENT")
print("=" * 65)

print("Python:", sys.version)
print("Executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA:", torch.version.cuda)

# --------------------------------------------------
# GPU INSPECTION
# --------------------------------------------------
print("\n" + "=" * 65)
print("GPU INFORMATION")
print("=" * 65)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
        print(f"Compute capability: {props.major}.{props.minor}")
else:
    print("No CUDA GPU detected!")

# --------------------------------------------------
# PACKAGE INSPECTION
# --------------------------------------------------
print("\n" + "=" * 65)
print("INSTALLED PACKAGES")
print("=" * 65)

packages = [
    "transformers",
    "accelerate",
    "peft",
    "bitsandbytes",
    "trl",
    "datasets",
    "tokenizers",
    "safetensors",
]

for package in packages:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# --------------------------------------------------
# DATASET INSPECTION
# --------------------------------------------------
print("\n" + "=" * 65)
print("DATASET")
print("=" * 65)

assert DATA_ROOT.exists(), f"Dataset not found: {DATA_ROOT}"
assert (DATA_ROOT / "DATASET_MANIFEST.json").exists()

for split in ["train", "validation", "test"]:
    path = LLM_DIR / f"sft_{split}.csv"

    assert path.exists(), f"Missing: {path}"

    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    print(f"\n{split.upper()}: {len(rows)} examples")
    print("Columns:", list(rows[0].keys()) if rows else "EMPTY")

    if rows:
        print("Example prompt:", rows[0]["prompt"][:250])
        print("Example completion:", rows[0]["completion"][:250])

        print("Average prompt chars:",
              round(sum(len(r["prompt"]) for r in rows) / len(rows)))

        print("Average completion chars:",
              round(sum(len(r["completion"]) for r in rows) / len(rows)))

print("\n" + "=" * 65)
print("ENVIRONMENT INSPECTION COMPLETE")
print("=" * 65)


PYTHON ENVIRONMENT
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Executable: /usr/bin/python3
PyTorch: 2.11.0+cu130
CUDA available: True
PyTorch CUDA: 13.0

GPU INFORMATION
GPU count: 2

GPU 0: Tesla T4
VRAM: 14.56 GB
Compute capability: 7.5

GPU 1: Tesla T4
VRAM: 14.56 GB
Compute capability: 7.5

INSTALLED PACKAGES
transformers: 5.20.2
accelerate: 1.15.0
peft: 0.21.2
bitsandbytes: 0.50.2
trl: NOT INSTALLED
datasets: 4.8.5
tokenizers: 0.23.3
safetensors: 0.8.0

DATASET

TRAIN: 912 examples
Columns: ['prompt', 'completion', 'task_type', 'mutation_id', 'profile_id', 'compound_id', 'evidence_ids', 'source_ids', 'review_status', 'annotation_confidence']
Example prompt: Explain the EGFR L858R mutation: its type, location, functional classification and relevance to EGFR inhibitors. Wild-type/mutant residues: L->R at position 858.
Example completion: EGFR L858R is a missense in the Protein kinase domain (UniProt P00533 numbering). It is classified as activating. Activation-loop a

In [3]:
!pip install -q bitsandbytes

In [4]:

import torch
import bitsandbytes as bnb

from transformers import BitsAndBytesConfig

print("bitsandbytes:", bnb.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# Test actual NF4 quantisation and dequantisation
x = torch.randn(128, 128, device="cuda", dtype=torch.float16)

q, state = bnb.functional.quantize_4bit(
    x,
    quant_type="nf4",
    compress_statistics=True
)

x_reconstructed = bnb.functional.dequantize_4bit(
    q,
    state
)

print("\nNF4 TEST")
print("Original shape:", x.shape)
print("Quantized shape:", q.shape)
print("Reconstructed shape:", x_reconstructed.shape)
print("Reconstruction error:",
      (x - x_reconstructed).abs().mean().item())

# Configuration for actual training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("\nQLoRA configuration created successfully.")


bitsandbytes: 0.50.2
CUDA: 13.0
GPU: Tesla T4

NF4 TEST
Original shape: torch.Size([128, 128])
Quantized shape: torch.Size([8192, 1])
Reconstructed shape: torch.Size([128, 128])
Reconstruction error: 0.07275390625

QLoRA configuration created successfully.


In [5]:

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# PATHS
# --------------------------------------------------
TRAIN_VAL_PATH = Path(
    "/kaggle/input/datasets/rudrakshjoshi5/"
    "updated-train-val-qwen/sft_train_plus_val.csv"
)

TEST_PATH = Path(
    "/kaggle/input/datasets/rudrakshjoshi5/"
    "ic-egfr-dataset/EGFR_RESEARCH_DATASET/"
    "mutation_scientist/sft_test.csv"
)

# --------------------------------------------------
# LOAD
# --------------------------------------------------
train_val_df = pd.read_csv(TRAIN_VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

required = {"prompt", "completion", "mutation_id", "task_type",
            "review_status", "annotation_confidence"}

assert required.issubset(train_val_df.columns)
assert required.issubset(test_df.columns)

print("=" * 65)
print("DATASET AUDIT")
print("=" * 65)

print("Combined train+validation:", len(train_val_df))
print("Held-out test:", len(test_df))

# --------------------------------------------------
# DATA QUALITY
# --------------------------------------------------
for name, df in [("TRAIN+VAL", train_val_df), ("TEST", test_df)]:
    print(f"\n--- {name} ---")

    print("Missing prompts:", df["prompt"].isna().sum())
    print("Missing completions:", df["completion"].isna().sum())

    print("Exact prompt-completion duplicates:",
          df.duplicated(["prompt", "completion"]).sum())

    print("\nTask distribution:")
    print(df["task_type"].value_counts(dropna=False))

    print("\nReview status:")
    print(df["review_status"].value_counts(dropna=False))

    print("\nMutation distribution:")
    print(df["mutation_id"].value_counts(dropna=False))

    print("\nPrompt length (characters):")
    print(df["prompt"].str.len().describe())

    print("\nCompletion length (characters):")
    print(df["completion"].str.len().describe())

# --------------------------------------------------
# MUTATION LEAKAGE
# --------------------------------------------------
train_mutations = set(train_val_df["mutation_id"].dropna())
test_mutations = set(test_df["mutation_id"].dropna())

print("\n" + "=" * 65)
print("MUTATION OVERLAP")
print("=" * 65)

print("Training mutations:", len(train_mutations))
print("Test mutations:", len(test_mutations))
print("Overlapping mutations:",
      sorted(train_mutations & test_mutations))

# --------------------------------------------------
# TASK COVERAGE
# --------------------------------------------------
print("\nTask types missing from train but present in test:")
print(sorted(
    set(test_df["task_type"].dropna()) -
    set(train_val_df["task_type"].dropna())
))

# --------------------------------------------------
# DUPLICATE TEST EXAMPLES
# --------------------------------------------------
train_pairs = set(zip(
    train_val_df["prompt"].fillna(""),
    train_val_df["completion"].fillna("")
))

test_pairs = set(zip(
    test_df["prompt"].fillna(""),
    test_df["completion"].fillna("")
))

print("\nExact train/test example overlap:",
      len(train_pairs & test_pairs))

print("\nAUDIT COMPLETE")


DATASET AUDIT
Combined train+validation: 1126
Held-out test: 215

--- TRAIN+VAL ---
Missing prompts: 0
Missing completions: 0
Exact prompt-completion duplicates: 0

Task distribution:
task_type
experimental_interpretation    417
inhibitor_comparison           263
evidence_qa                    206
assay_distinction              107
mutation_classification         47
mutation_mechanism              43
resistance_interpretation       40
experiment_design                1
hypothesis_generation            1
evidence_limitations             1
Name: count, dtype: int64

Review status:
review_status
teacher_generated_unverified_autochecked    1041
model_generated_grounded                      85
Name: count, dtype: int64

Mutation distribution:
mutation_id
EGFR_C797S                  146
EGFR_L858R_T790M_C797S      125
EGFR_ex19del                100
EGFR_T790M                   99
EGFR_L858R_C797S             72
EGFR_L858R_T790M             71
EGFR_ex19del_T790M           71
EGFR_WT         

In [6]:

import torch
import bitsandbytes as bnb

print("bitsandbytes:", bnb.__version__)
print("CUDA:", torch.version.cuda)

x = torch.randn(128, 128, device="cuda:0", dtype=torch.float16)

q, state = bnb.functional.quantize_4bit(
    x,
    quant_type="nf4",
    compress_statistics=True
)

x_hat = bnb.functional.dequantize_4bit(q, state)

print("NF4 quantisation: SUCCESS")
print("Mean reconstruction error:",
      (x - x_hat).abs().mean().item())
print("GPU:", torch.cuda.get_device_name(0))

del x, q, state, x_hat
torch.cuda.empty_cache()


bitsandbytes: 0.50.2
CUDA: 13.0
NF4 quantisation: SUCCESS
Mean reconstruction error: 0.0726318359375
GPU: Tesla T4


In [7]:

import torch
import bitsandbytes as bnb

from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    Qwen3_5ForConditionalGeneration
)

MODEL_ID = "Qwen/Qwen3.5-4B"

# --------------------------------------------------
# QUANTISATION CONFIGURATION
# --------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# --------------------------------------------------
# TOKENIZER
# --------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded")
print("EOS:", tokenizer.eos_token)
print("PAD:", tokenizer.pad_token)

# --------------------------------------------------
# MODEL
# --------------------------------------------------
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True
)

model.config.use_cache = False
model.config.text_config.use_cache = False

print("\nModel loaded successfully")
print("Model class:", type(model).__name__)
print("Language backbone:", type(model.model.language_model).__name__)
print("Vision encoder:", type(model.model.visual).__name__)

print("\nGPU memory:")
print(
    f"Allocated: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB"
)
print(
    f"Reserved: {torch.cuda.memory_reserved(0)/1024**3:.2f} GB"
)


Tokenizer loaded
EOS: <|im_end|>
PAD: <|endoftext|>


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]


Model loaded successfully
Model class: Qwen3_5ForConditionalGeneration
Language backbone: Qwen3_5TextModel
Vision encoder: Qwen3_5VisionModel

GPU memory:
Allocated: 3.08 GB
Reserved: 3.12 GB


In [8]:

import torch
import bitsandbytes as bnb
from collections import Counter

language_prefix = "model.language_model."

language_linear_modules = []

for name, module in model.named_modules():
    if (
        name.startswith(language_prefix)
        and isinstance(module, torch.nn.Linear)
    ):
        language_linear_modules.append(name)

print("Language-model linear modules:", len(language_linear_modules))

print("\nMODULE BREAKDOWN")
print("=" * 65)

module_groups = Counter(
    name.split(".")[-1]
    for name in language_linear_modules
)

for name, count in module_groups.most_common():
    print(f"{name:30s} {count}")

print("\nSAMPLE MODULE PATHS")
print("=" * 65)

for name in language_linear_modules[:35]:
    print(name)

# Verify vision encoder is excluded from our target list
assert all(
    "visual" not in name
    for name in language_linear_modules
)

assert len(language_linear_modules) > 0

print("\nArchitecture inspection successful.")


Language-model linear modules: 248

MODULE BREAKDOWN
gate_proj                      32
up_proj                        32
down_proj                      32
out_proj                       24
in_proj_qkv                    24
in_proj_z                      24
in_proj_b                      24
in_proj_a                      24
q_proj                         8
k_proj                         8
v_proj                         8
o_proj                         8

SAMPLE MODULE PATHS
model.language_model.layers.0.linear_attn.out_proj
model.language_model.layers.0.linear_attn.in_proj_qkv
model.language_model.layers.0.linear_attn.in_proj_z
model.language_model.layers.0.linear_attn.in_proj_b
model.language_model.layers.0.linear_attn.in_proj_a
model.language_model.layers.0.mlp.gate_proj
model.language_model.layers.0.mlp.up_proj
model.language_model.layers.0.mlp.down_proj
model.language_model.layers.1.linear_attn.out_proj
model.language_model.layers.1.linear_attn.in_proj_qkv
model.language_model.layer

In [9]:

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# --------------------------------------------------
# PREPARE QUANTIZED MODEL
# --------------------------------------------------
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

# --------------------------------------------------
# LORA CONFIGURATION
# --------------------------------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=language_linear_modules,
    task_type="CAUSAL_LM"
)

# --------------------------------------------------
# INJECT ADAPTERS
# --------------------------------------------------
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("\nQLoRA adapters successfully injected.")

# Check that only language-model modules have adapters
adapter_modules = [
    name for name, param in model.named_parameters()
    if param.requires_grad
]

assert len(adapter_modules) > 0

print("Trainable parameter tensors:", len(adapter_modules))
print("Example trainable parameters:")

for name in adapter_modules[:15]:
    print(name)


trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101

QLoRA adapters successfully injected.
Trainable parameter tensors: 496
Example trainable parameters:
base_model.model.model.language_model.layers.0.linear_attn.out_proj.lora_A.default.weight
base_model.model.model.language_model.layers.0.linear_attn.out_proj.lora_B.default.weight
base_model.model.model.language_model.layers.0.linear_attn.in_proj_qkv.lora_A.default.weight
base_model.model.model.language_model.layers.0.linear_attn.in_proj_qkv.lora_B.default.weight
base_model.model.model.language_model.layers.0.linear_attn.in_proj_z.lora_A.default.weight
base_model.model.model.language_model.layers.0.linear_attn.in_proj_z.lora_B.default.weight
base_model.model.model.language_model.layers.0.linear_attn.in_proj_b.lora_A.default.weight
base_model.model.model.language_model.layers.0.linear_attn.in_proj_b.lora_B.default.weight
base_model.model.model.language_model.layers.0.linear_attn.in_proj_a.lora_A.default.weig

In [10]:

import pandas as pd
import numpy as np

SEED = 42
MAX_LENGTH = 1024

# train_val_df should already contain the updated 1,126-row CSV.
assert len(train_val_df) == 1126
assert {"prompt", "completion"}.issubset(train_val_df.columns)

# Shuffle and split reproducibly
shuffled = train_val_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

n_val = round(len(shuffled) * 0.10)

validation_df = shuffled.iloc[:n_val].reset_index(drop=True)
training_df = shuffled.iloc[n_val:].reset_index(drop=True)

print("TRAINING:", len(training_df))
print("VALIDATION:", len(validation_df))
print("TEST:", len(test_df))

assert len(training_df) + len(validation_df) == len(train_val_df)
assert len(test_df) == 215

print("\nTraining task distribution:")
print(training_df["task_type"].value_counts())

print("\nValidation task distribution:")
print(validation_df["task_type"].value_counts())


TRAINING: 1013
VALIDATION: 113
TEST: 215

Training task distribution:
task_type
experimental_interpretation    368
inhibitor_comparison           243
evidence_qa                    188
assay_distinction               93
mutation_classification         42
resistance_interpretation       38
mutation_mechanism              38
experiment_design                1
hypothesis_generation            1
evidence_limitations             1
Name: count, dtype: int64

Validation task distribution:
task_type
experimental_interpretation    49
inhibitor_comparison           20
evidence_qa                    18
assay_distinction              14
mutation_classification         5
mutation_mechanism              5
resistance_interpretation       2
Name: count, dtype: int64


In [11]:
import pandas as pd
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-4B"

CSV_PATH = "/kaggle/input/datasets/rudrakshjoshi5/updated-train-val-qwen/sft_train_plus_val.csv"

train_val_df = pd.read_csv(CSV_PATH)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    padding_side="right"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sample = train_val_df.iloc[0]
prompt = str(sample["prompt"])
completion = str(sample["completion"])

prompt_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=False,
    enable_thinking=False
)

full_ids = tokenizer.apply_chat_template(
    [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": completion}
    ],
    tokenize=True,
    add_generation_prompt=False,
    return_dict=False,
    enable_thinking=False
)

print("Dataset:", train_val_df.shape)
print("Prompt length:", len(prompt_ids))
print("Full length:", len(full_ids))
print("Exact prefix:", full_ids[:len(prompt_ids)] == prompt_ids)
print("\nDecoded completion:")
print(tokenizer.decode(full_ids[len(prompt_ids):]))

Dataset: (1126, 10)
Prompt length: 54
Full length: 122
Exact prefix: True

Decoded completion:
EGFR L858R is a missense in the Protein kinase domain (UniProt P00533 numbering). It is classified as activating. Activation-loop activating mutation; TKI-sensitising. Biochemical or cellular validation is required to quantify the magnitude of any sensitivity or resistance in a given context.<|im_end|>



In [12]:

prompt = str(sample["prompt"])
completion = str(sample["completion"])

prompt_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=False
)

full_ids = tokenizer.apply_chat_template(
    [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": completion}
    ],
    tokenize=True,
    add_generation_prompt=False,
    return_dict=False
)

print("Prompt type:", type(prompt_ids))
print("Full type:", type(full_ids))
print("Prompt length:", len(prompt_ids))
print("Full length:", len(full_ids))

# First differing token
common = 0
for a, b in zip(prompt_ids, full_ids):
    if a != b:
        break
    common += 1

print("\nCommon prefix:", common)
print("Exact prefix:", full_ids[:len(prompt_ids)] == prompt_ids)

print("\nPROMPT IDS (last 30):")
print(prompt_ids[-30:])

print("\nFULL IDS (around divergence):")
print(full_ids[max(0, common-10):common+30])

print("\nPROMPT DECODE:")
print(repr(tokenizer.decode(prompt_ids[-40:])))

print("\nFULL DECODE:")
print(repr(tokenizer.decode(full_ids[max(0, common-15):common+40])))

print("\nCHAT TEMPLATE:")
print(tokenizer.chat_template)


Prompt type: <class 'list'>
Full type: <class 'list'>
Prompt length: 52
Full length: 122

Common prefix: 51
Exact prefix: False

PROMPT IDS (last 30):
[321, 39512, 310, 54226, 10191, 65264, 13, 13237, 10289, 3082, 243816, 68290, 25, 436, 397, 49, 506, 2234, 220, 23, 20, 23, 13, 248046, 198, 248045, 74455, 198, 248068, 198]

FULL IDS (around divergence):
[23, 20, 23, 13, 248046, 198, 248045, 74455, 198, 248068, 271, 248069, 271, 9097, 10191, 436, 23, 20, 23, 49, 369, 264, 3016, 1081, 303, 279, 46763, 78266, 7697, 318, 88567, 12067, 387, 15, 15, 20, 18, 18, 75579, 553]

PROMPT DECODE:
'R mutation: its type, location, functional classification and relevance to EGFR inhibitors. Wild-type/mutant residues: L->R at position 858.<|im_end|>\n<|im_start|>assistant\n<think>\n'

FULL DECODE:
'->R at position 858.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nEGFR L858R is a missense in the Protein kinase domain (UniProt P00533 numbering). It is classified as activating. Activation-loop 

In [13]:

def encode_example(prompt, completion):
    prompt_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": str(prompt)}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=False,
        enable_thinking=False
    )

    full_ids = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": str(prompt)},
            {"role": "assistant", "content": str(completion)}
        ],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False,
        enable_thinking=False
    )

    prompt_length = len(prompt_ids)

    if full_ids[:prompt_length] != prompt_ids:
        raise ValueError(
            f"Prefix mismatch: prompt={prompt_length}, "
            f"common={sum(a == b for a, b in zip(prompt_ids, full_ids))}"
        )

    # Truncate to maximum context length
    full_ids = full_ids[:MAX_LENGTH]

    labels = full_ids.copy()

    # Mask prompt; supervise assistant answer only
    labels[:prompt_length] = [-100] * prompt_length

    if not any(token != -100 for token in labels):
        return None

    return {
        "input_ids": full_ids,
        "labels": labels,
        "prompt_length": prompt_length,
        "sequence_length": len(full_ids)
    }


# Test
sample_encoded = encode_example(
    sample["prompt"],
    sample["completion"]
)

print("Prompt tokens:", sample_encoded["prompt_length"])
print("Total tokens:", sample_encoded["sequence_length"])
print("Supervised tokens:", sum(
    x != -100 for x in sample_encoded["labels"]
))

print("\nSUPERVISED TEXT:")
print(tokenizer.decode([
    x for x in sample_encoded["labels"]
    if x != -100
]))


Prompt tokens: 54
Total tokens: 122
Supervised tokens: 68

SUPERVISED TEXT:
EGFR L858R is a missense in the Protein kinase domain (UniProt P00533 numbering). It is classified as activating. Activation-loop activating mutation; TKI-sensitising. Biochemical or cellular validation is required to quantify the magnitude of any sensitivity or resistance in a given context.<|im_end|>



In [14]:

def encode_dataframe(df, split_name):
    encoded = []
    skipped = 0
    potentially_truncated = 0

    for _, row in df.iterrows():
        item = encode_example(
            row["prompt"],
            row["completion"]
        )

        if item is None:
            skipped += 1
            continue

        if item["sequence_length"] == MAX_LENGTH:
            potentially_truncated += 1

        # Preserve metadata for later evaluation, not model input.
        item["task_type"] = row.get("task_type", None)
        item["mutation_id"] = row.get("mutation_id", None)
        item["profile_id"] = row.get("profile_id", None)
        item["review_status"] = row.get("review_status", None)

        encoded.append(item)

    lengths = [x["sequence_length"] for x in encoded]
    supervised = [
        sum(token != -100 for token in x["labels"])
        for x in encoded
    ]

    print(f"\n{split_name}")
    print("-" * 45)
    print("Encoded:", len(encoded))
    print("Skipped:", skipped)
    print("Sequences at MAX_LENGTH:", potentially_truncated)

    if encoded:
        print(f"Mean sequence length: {np.mean(lengths):.1f}")
        print(f"Maximum sequence length: {max(lengths)}")
        print(f"Mean supervised tokens: {np.mean(supervised):.1f}")

    return encoded


train_encoded = encode_dataframe(training_df, "TRAIN")
val_encoded = encode_dataframe(validation_df, "VALIDATION")

print("\nSPLIT SUMMARY")
print("Training:", len(train_encoded))
print("Validation:", len(val_encoded))
print("Original test rows:", len(test_df))



TRAIN
---------------------------------------------
Encoded: 1013
Skipped: 0
Sequences at MAX_LENGTH: 0
Mean sequence length: 209.7
Maximum sequence length: 429
Mean supervised tokens: 125.7

VALIDATION
---------------------------------------------
Encoded: 113
Skipped: 0
Sequences at MAX_LENGTH: 0
Mean sequence length: 205.8
Maximum sequence length: 417
Mean supervised tokens: 123.1

SPLIT SUMMARY
Training: 1013
Validation: 113
Original test rows: 215


In [15]:

import torch
from torch.utils.data import Dataset, DataLoader

class EGFRDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples[idx]
        return {
            "input_ids": item["input_ids"],
            "labels": item["labels"]
        }


class CompletionOnlyCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, batch):
        max_len = max(len(x["input_ids"]) for x in batch)

        input_ids = []
        labels = []
        attention_masks = []

        for item in batch:
            padding = max_len - len(item["input_ids"])

            input_ids.append(
                item["input_ids"] + [self.pad_token_id] * padding
            )

            labels.append(
                item["labels"] + [-100] * padding
            )

            attention_masks.append(
                [1] * len(item["input_ids"]) + [0] * padding
            )

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(
                attention_masks, dtype=torch.long
            ),
            "labels": torch.tensor(labels, dtype=torch.long)
        }


train_dataset = EGFRDataset(train_encoded)
val_dataset = EGFRDataset(val_encoded)

collator = CompletionOnlyCollator(tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=collator,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Training batches: 1013
Validation batches: 113


In [16]:

import gc

gc.collect()
torch.cuda.empty_cache()

batch = next(iter(train_loader))

batch = {
    key: value.to("cuda:0")
    for key, value in batch.items()
}

model.eval()

with torch.no_grad():
    outputs = model(**batch)

print("Input shape:", tuple(batch["input_ids"].shape))
print("Label shape:", tuple(batch["labels"].shape))
print("Loss:", outputs.loss.item())
print("Finite loss:", torch.isfinite(outputs.loss).item())

assert torch.isfinite(outputs.loss)
assert outputs.loss.item() > 0

print("\nForward pass successful.")

del batch, outputs
gc.collect()
torch.cuda.empty_cache()


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Input shape: (1, 197)
Label shape: (1, 197)
Loss: 2.0038564205169678
Finite loss: True

Forward pass successful.


In [17]:

import math
import os
import gc
import random
import numpy as np
import torch
import bitsandbytes as bnb

from transformers import get_cosine_schedule_with_warmup

SEED = 42
EPOCHS = 3
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Only optimise trainable LoRA parameters
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = bnb.optim.PagedAdamW8bit(
    trainable_params,
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    weight_decay=WEIGHT_DECAY
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_steps * 0.05))

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.amp.GradScaler("cuda")

OUTPUT_DIR = "/kaggle/working/egfr_qwen35_qlora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Trainable parameters:", sum(p.numel() for p in trainable_params))
print("Steps per epoch:", steps_per_epoch)
print("Total optimiser steps:", total_steps)
print("Warmup steps:", warmup_steps)
print("Output directory:", OUTPUT_DIR)

print(
    f"GPU allocated: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB"
)


Trainable parameters: 32464896
Steps per epoch: 127
Total optimiser steps: 381
Warmup steps: 19
Output directory: /kaggle/working/egfr_qwen35_qlora
GPU allocated: 4.40 GB


In [18]:

def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            batch = {
                k: v.to("cuda:0", non_blocking=True)
                for k, v in batch.items()
            }

            supervised_tokens = (
                batch["labels"] != -100
            ).sum().item()

            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16
            ):
                outputs = model(**batch)

            total_loss += outputs.loss.item() * supervised_tokens
            total_tokens += supervised_tokens

            del batch, outputs

    model.train()

    if total_tokens == 0:
        raise ValueError("Validation contains no supervised tokens.")

    return total_loss / total_tokens


In [19]:

from tqdm.auto import tqdm
import pandas as pd

best_val_loss = float("inf")
patience = 2
epochs_without_improvement = 0
history = []

optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    model.train()

    running_loss = 0.0
    processed = 0

    progress = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for step, batch in progress:
        batch = {
            k: v.to("cuda:0", non_blocking=True)
            for k, v in batch.items()
        }

        # Handle the final incomplete accumulation window correctly
        window_start = (step // GRAD_ACCUM) * GRAD_ACCUM
        window_size = min(
            GRAD_ACCUM,
            len(train_loader) - window_start
        )

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch+1}, step {step}"
            )

        running_loss += loss.item()
        processed += 1

        scaled_loss = loss / window_size
        scaler.scale(scaled_loss).backward()

        # Optimiser update at accumulation boundary
        is_update_step = (
            (step + 1) % GRAD_ACCUM == 0
            or (step + 1) == len(train_loader)
        )

        if is_update_step:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                MAX_GRAD_NORM
            )

            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

        del batch, outputs, loss, scaled_loss

    train_loss = running_loss / processed

    # Validation
    val_loss = evaluate(model, val_loader)

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Training loss:   {train_loss:.5f}")
    print(f"Validation loss: {val_loss:.5f}")
    print(
        f"GPU peak memory: "
        f"{torch.cuda.max_memory_allocated(0)/1024**3:.2f} GB"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "learning_rate": scheduler.get_last_lr()[0]
    })

    # Save best adapter
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        model.save_pretrained(
            os.path.join(OUTPUT_DIR, "best_adapter")
        )
        tokenizer.save_pretrained(
            os.path.join(OUTPUT_DIR, "best_adapter")
        )

        print("New best checkpoint saved.")

    else:
        epochs_without_improvement += 1
        print(
            f"No improvement: "
            f"{epochs_without_improvement}/{patience}"
        )

    pd.DataFrame(history).to_csv(
        os.path.join(OUTPUT_DIR, "training_history.csv"),
        index=False
    )

    gc.collect()
    torch.cuda.empty_cache()

    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

print("\nTRAINING COMPLETE")
print("Best validation loss:", best_val_loss)
print("Saved at:", os.path.join(OUTPUT_DIR, "best_adapter"))


Epoch 1/3:   0%|          | 0/1013 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:

import matplotlib.pyplot as plt

history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 5))
plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
    label="Training loss"
)
plt.plot(
    history_df["epoch"],
    history_df["validation_loss"],
    marker="o",
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("EGFR MutationScientist QLoRA Training")
plt.xticks(history_df["epoch"])
plt.legend()
plt.grid(alpha=0.3)
plt.show()

display(history_df)


In [20]:
import os
import gc
import torch

def clean_encoded_data(rows):
    if hasattr(rows, "to_dict"):
        rows = rows.to_dict("records")

    cleaned = []
    for row in rows:
        cleaned.append({
            "input_ids": [int(x) for x in row["input_ids"]],
            "labels": [int(x) for x in row["labels"]],
        })
    return cleaned

train_data = clean_encoded_data(train_encoded)
val_data = clean_encoded_data(val_encoded)

DATA_PATH = "/kaggle/working/egfr_encoded_data.pt"

torch.save({
    "train": train_data,
    "validation": val_data,
}, DATA_PATH)

print("Saved:", DATA_PATH)
print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Example sequence length:", len(train_data[0]["input_ids"]))

Saved: /kaggle/working/egfr_encoded_data.pt
Training examples: 1013
Validation examples: 113
Example sequence length: 257


In [21]:
for name in [
    "model", "optimizer", "scheduler", "scaler",
    "outputs", "batch", "train_loader", "val_loader",
    "train_dataset", "val_dataset"
]:
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

!nvidia-smi

Sun Nov 15 02:46:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             31W /   70W |    6029MiB /  15360MiB |     27%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
%%writefile /kaggle/working/train_ddp.py

import os
import gc
import json
import math
import random
import argparse
from contextlib import nullcontext

import numpy as np
import torch
import torch.distributed as dist
import bitsandbytes as bnb

from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP

from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    Qwen3_5ForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_ID = "Qwen/Qwen3.5-4B"

DATA_PATH = "/kaggle/working/egfr_encoded_data.pt"
OUTPUT_DIR = "/kaggle/working/egfr_qwen35_qlora_ddp"

SEED = 42
EPOCHS = 3

MICRO_BATCH_SIZE = 1
GRAD_ACCUM = 4

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

NUM_WORKERS = 0


# ============================================================
# DISTRIBUTED SETUP
# ============================================================

def setup_distributed():
    dist.init_process_group(backend="nccl")

    local_rank = int(os.environ["LOCAL_RANK"])
    rank = dist.get_rank()
    world_size = dist.get_world_size()

    torch.cuda.set_device(local_rank)

    return local_rank, rank, world_size


def cleanup_distributed():
    dist.destroy_process_group()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ============================================================
# DATASET AND COLLATOR
# ============================================================

class EGFRDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


class CompletionOnlyCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        batch_size = len(batch)

        input_ids = torch.full(
            (batch_size, max_len),
            self.pad_token_id,
            dtype=torch.long,
        )

        labels = torch.full(
            (batch_size, max_len),
            -100,
            dtype=torch.long,
        )

        attention_mask = torch.zeros(
            (batch_size, max_len),
            dtype=torch.long,
        )

        for i, item in enumerate(batch):
            length = len(item["input_ids"])

            input_ids[i, :length] = torch.tensor(
                item["input_ids"], dtype=torch.long
            )

            labels[i, :length] = torch.tensor(
                item["labels"], dtype=torch.long
            )

            attention_mask[i, :length] = 1

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


# ============================================================
# MODEL LOADING
# ============================================================

def load_model(local_rank):

    print(f"[GPU {local_rank}] Loading tokenizer...", flush=True)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        padding_side="right",
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"[GPU {local_rank}] Loading quantised model...", flush=True)

    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": local_rank},
        low_cpu_mem_usage=True,
    )

    model.config.use_cache = False
    model.config.text_config.use_cache = False

    # Target every Linear module in the language backbone,
    # matching the successful single-GPU LoRA configuration.
    language_linear_modules = [
        name
        for name, module in model.named_modules()
        if isinstance(module, torch.nn.Linear)
        and name.startswith("model.language_model.")
    ]

    if not language_linear_modules:
        raise RuntimeError("No language-model Linear modules found.")

    print(
        f"[GPU {local_rank}] LoRA targets: "
        f"{len(language_linear_modules)}",
        flush=True,
    )

    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        },
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        target_modules=language_linear_modules,
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)

    model.print_trainable_parameters()

    return model, tokenizer


# ============================================================
# VALIDATION
# ============================================================

@torch.no_grad()
def evaluate(model, val_loader, device):

    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in val_loader:

        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            outputs = model(**batch)

        valid_tokens = batch["labels"].ne(-100).sum().item()

        total_loss += outputs.loss.item() * valid_tokens
        total_tokens += valid_tokens

    model.train()

    return total_loss / max(total_tokens, 1)


# ============================================================
# MAIN
# ============================================================

def main():

    local_rank, rank, world_size = setup_distributed()
    device = torch.device(f"cuda:{local_rank}")

    set_seed(SEED + rank)

    if rank == 0:
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    dist.barrier()

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    data = torch.load(
        DATA_PATH,
        map_location="cpu",
        weights_only=False,
    )

    train_dataset = EGFRDataset(data["train"])
    val_dataset = EGFRDataset(data["validation"])

    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=SEED,
        drop_last=False,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=MICRO_BATCH_SIZE,
        sampler=train_sampler,
        collate_fn=None,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
    )

    # --------------------------------------------------------
    # Load model independently on each GPU
    # --------------------------------------------------------

    model, tokenizer = load_model(local_rank)

    collator = CompletionOnlyCollator(tokenizer.pad_token_id)
    train_loader.collate_fn = collator

    # Only rank 0 performs validation, using the complete
    # validation set without distributed-sampler duplication.
    val_loader = None

    if rank == 0:
        val_loader = DataLoader(
            val_dataset,
            batch_size=1,
            shuffle=False,
            collate_fn=collator,
            num_workers=0,
            pin_memory=True,
        )

    # --------------------------------------------------------
    # Wrap with DDP
    # --------------------------------------------------------

    model = DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank,
        find_unused_parameters=False,
        broadcast_buffers=False,
        gradient_as_bucket_view=True,
    )

    trainable_params = [
        p for p in model.parameters() if p.requires_grad
    ]

    optimizer = bnb.optim.PagedAdamW8bit(
        trainable_params,
        lr=LEARNING_RATE,
        betas=(0.9, 0.999),
        weight_decay=WEIGHT_DECAY,
    )

    steps_per_epoch = math.ceil(
        len(train_loader) / GRAD_ACCUM
    )

    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = max(1, int(total_steps * 0.05))

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = torch.amp.GradScaler("cuda")

    if rank == 0:
        print("\n========== TRAINING CONFIGURATION ==========")
        print("World size:", world_size)
        print("Training examples:", len(train_dataset))
        print("Validation examples:", len(val_dataset))
        print("Micro-batch per GPU:", MICRO_BATCH_SIZE)
        print("Gradient accumulation:", GRAD_ACCUM)
        print("Effective global batch:", MICRO_BATCH_SIZE * GRAD_ACCUM * world_size)
        print("Steps per epoch:", steps_per_epoch)
        print("Total optimiser steps:", total_steps)
        print("Warmup steps:", warmup_steps)
        print("============================================\n")

    best_val_loss = float("inf")
    history = []

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(EPOCHS):

        train_sampler.set_epoch(epoch)
        model.train()

        optimizer.zero_grad(set_to_none=True)

        epoch_loss = 0.0
        epoch_tokens = 0

        num_batches = len(train_loader)

        for step, batch in enumerate(train_loader):

            batch = {
                key: value.to(device, non_blocking=True)
                for key, value in batch.items()
            }

            group_start = (step // GRAD_ACCUM) * GRAD_ACCUM
            group_end = min(group_start + GRAD_ACCUM, num_batches)
            group_size = group_end - group_start

            is_sync_step = (step + 1 == group_end)

            sync_context = (
                nullcontext()
                if is_sync_step
                else model.no_sync()
            )

            with sync_context:

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    outputs = model(**batch)
                    loss = outputs.loss

                valid_tokens = batch["labels"].ne(-100).sum().item()

                epoch_loss += loss.detach().item() * valid_tokens
                epoch_tokens += valid_tokens

                scaled_loss = loss / group_size
                scaler.scale(scaled_loss).backward()

            if is_sync_step:

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    trainable_params,
                    MAX_GRAD_NORM,
                )

                scaler.step(optimizer)
                scaler.update()

                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            if rank == 0 and (step + 1) % 50 == 0:
                print(
                    f"Epoch {epoch + 1}/{EPOCHS} | "
                    f"Batch {step + 1}/{num_batches} | "
                    f"Loss {loss.item():.4f} | "
                    f"LR {scheduler.get_last_lr()[0]:.2e}",
                    flush=True,
                )

        # ----------------------------------------------------
        # Aggregate training loss across GPUs
        # ----------------------------------------------------

        stats = torch.tensor(
            [epoch_loss, epoch_tokens],
            dtype=torch.float64,
            device=device,
        )

        dist.all_reduce(stats, op=dist.ReduceOp.SUM)

        train_loss = (stats[0] / stats[1]).item()

        # ----------------------------------------------------
        # Validation on rank 0
        # ----------------------------------------------------

        dist.barrier()

        if rank == 0:

            val_loss = evaluate(
                model.module,
                val_loader,
                device,
            )

            print(
                f"\nEpoch {epoch + 1}/{EPOCHS} complete | "
                f"Train loss: {train_loss:.5f} | "
                f"Validation loss: {val_loss:.5f}",
                flush=True,
            )

            history.append({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "validation_loss": val_loss,
            })

            if val_loss < best_val_loss:

                best_val_loss = val_loss

                best_dir = os.path.join(
                    OUTPUT_DIR,
                    "best_adapter",
                )

                model.module.save_pretrained(best_dir)
                tokenizer.save_pretrained(best_dir)

                print(
                    f"Saved new best adapter: {best_dir}",
                    flush=True,
                )

            with open(
                os.path.join(OUTPUT_DIR, "training_history.json"),
                "w",
            ) as f:
                json.dump(history, f, indent=2)

        dist.barrier()

        gc.collect()
        torch.cuda.empty_cache()

    if rank == 0:
        print("\nTraining completed.")
        print("Best validation loss:", best_val_loss)
        print("Best adapter:", os.path.join(OUTPUT_DIR, "best_adapter"))

    dist.barrier()
    cleanup_distributed()


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_ddp.py


In [23]:
%env TOKENIZERS_PARALLELISM=false
%env OMP_NUM_THREADS=4

!python -m torch.distributed.run \
    --standalone \
    --nnodes=1 \
    --nproc_per_node=2 \
    /kaggle/working/train_ddp.py

env: TOKENIZERS_PARALLELISM=false
env: OMP_NUM_THREADS=4
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1115 02:47:20.408506787 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
[GPU 0] Loading tokenizer...
[GPU 1] Loading tokenizer...
[GPU 0] Loading quantised model...
[GPU 1] Loading quantised model...
Loading weights: 100%|████████████████████████| 723/723 [00:11<00:00, 60.47it/s]
[GPU 1] LoRA targets: 248
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101

========== TRAINING CONFIGURATION ==========
World size: 2
Training examples: 1013
Val

In [24]:
print(train_val_df.columns.tolist())

['prompt', 'completion', 'task_type', 'mutation_id', 'profile_id', 'compound_id', 'evidence_ids', 'source_ids', 'review_status', 'annotation_confidence']


In [25]:
import pandas as pd
import numpy as np

df = train_val_df.copy()

print("Total examples:", len(df))
print("Unique mutations:", df["mutation_id"].nunique())
print("Missing mutation IDs:", df["mutation_id"].isna().sum())
print("Examples per mutation:")
print(df.groupby("mutation_id").size().describe())

print("\nLargest mutation groups:")
print(df.groupby("mutation_id").size().sort_values(ascending=False).head(10))

Total examples: 1126
Unique mutations: 15
Missing mutation IDs: 51
Examples per mutation:
count     15.000000
mean      71.666667
std       35.806358
min       14.000000
25%       62.000000
50%       69.000000
75%       85.500000
max      146.000000
dtype: float64

Largest mutation groups:
mutation_id
EGFR_C797S                146
EGFR_L858R_T790M_C797S    125
EGFR_ex19del              100
EGFR_T790M                 99
EGFR_L858R_C797S           72
EGFR_ex19del_T790M         71
EGFR_L858R_T790M           71
EGFR_WT                    69
EGFR_ex19del_C797S         68
EGFR_L858R                 65
dtype: int64


In [26]:
missing = train_val_df[train_val_df["mutation_id"].isna()].copy()

print("Missing mutation rows:", len(missing))

print("\nProfile IDs:")
print(missing["profile_id"].value_counts(dropna=False).head(20))

print("\nCompound IDs:")
print(missing["compound_id"].value_counts(dropna=False).head(20))

print("\nTask types:")
print(missing["task_type"].value_counts(dropna=False))

print("\nSample prompts:")
for i, row in missing.head(12).iterrows():
    print(f"\n--- Row {i} ---")
    print("Profile:", row["profile_id"])
    print("Compound:", row["compound_id"])
    print("Prompt:", str(row["prompt"])[:500])

Missing mutation rows: 51

Profile IDs:
profile_id
NaN    51
Name: count, dtype: int64

Compound IDs:
compound_id
CMP_BLU945          18
CMP_OSIMERTINIB     17
CMP_SILEVERTINIB    13
NaN                  1
CMP_JBJ09063         1
                     1
Name: count, dtype: int64

Task types:
task_type
assay_distinction              17
experimental_interpretation    17
inhibitor_comparison           10
resistance_interpretation       6
evidence_limitations            1
Name: count, dtype: int64

Sample prompts:

--- Row 69 ---
Profile: nan
Compound: nan
Prompt: A dataset preserves KD, Ki and IC50 values separately for EGFR–inhibitor pairs and does not pool them. Explain why these measurement types must not be treated as interchangeable.

--- Row 70 ---
Profile: nan
Compound: CMP_BLU945
Prompt: For BLU-945 against wild-type EGFR, the biochemical IC50 is 86.0 nM at Km,ATP but 1490 nM at 1 mmol/L ATP (Melis 2026). Explain why the two values differ and the implication for comparing inhibitors

In [27]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer
from sklearn.model_selection import GroupShuffleSplit

MODEL_ID = "Qwen/Qwen3.5-4B"
CSV_PATH = "/kaggle/input/datasets/rudrakshjoshi5/updated-train-val-qwen/sft_train_plus_val.csv"

OUTPUT_DIR = "/kaggle/working/egfr_qwen35_optuna_ddp"
DATA_DIR = f"{OUTPUT_DIR}/data"

os.makedirs(DATA_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
df["mutation_group"] = df["mutation_id"].fillna("UNKNOWN_MUTATION").astype(str)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 1024

def encode_example(prompt, completion):
    prompt_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": str(prompt)}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=False,
        enable_thinking=False
    )

    full_ids = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": str(prompt)},
            {"role": "assistant", "content": str(completion)}
        ],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False,
        enable_thinking=False
    )

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("Chat-template prefix mismatch.")

    full_ids = full_ids[:MAX_LENGTH]
    prompt_length = min(len(prompt_ids), len(full_ids))

    labels = full_ids.copy()
    labels[:prompt_length] = [-100] * prompt_length

    if not any(x != -100 for x in labels):
        return None

    return {
        "input_ids": full_ids,
        "labels": labels
    }

print("Encoding dataset...")

encoded = []
for _, row in df.iterrows():
    item = encode_example(row["prompt"], row["completion"])
    if item is not None:
        item["mutation_group"] = row["mutation_group"]
        encoded.append(item)

encoded_df = pd.DataFrame(encoded)

# Group-aware 90/10 final train/validation split.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        encoded_df,
        groups=encoded_df["mutation_group"]
    )
)

train_pool = encoded_df.iloc[train_idx].reset_index(drop=True)
holdout = encoded_df.iloc[val_idx].reset_index(drop=True)

assert set(train_pool["mutation_group"]).isdisjoint(
    set(holdout["mutation_group"])
)

def save_rows(frame, path):
    rows = frame[["input_ids", "labels"]].to_dict("records")
    torch.save(rows, path)

save_rows(train_pool, f"{DATA_DIR}/final_train.pt")
save_rows(holdout, f"{DATA_DIR}/final_val.pt")

print("\nEncoding complete.")
print("Total encoded:", len(encoded_df))
print("Training pool:", len(train_pool))
print("Final validation:", len(holdout))
print("Training mutation groups:", train_pool["mutation_group"].nunique())
print("Validation mutation groups:", holdout["mutation_group"].nunique())
print("Unknown mutation rows:", (encoded_df["mutation_group"] == "UNKNOWN_MUTATION").sum())
print("Output:", OUTPUT_DIR)

Encoding dataset...

Encoding complete.
Total encoded: 1126
Training pool: 945
Final validation: 181
Training mutation groups: 14
Validation mutation groups: 2
Unknown mutation rows: 51
Output: /kaggle/working/egfr_qwen35_optuna_ddp


In [28]:
%%writefile /kaggle/working/egfr_ddp_worker.py

import os
import gc
import json
import math
import argparse
from contextlib import nullcontext

import torch
import torch.distributed as dist
import bitsandbytes as bnb

from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP

from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    Qwen3_5ForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


class EGFRDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


class CompletionOnlyCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        n = len(batch)

        input_ids = torch.full(
            (n, max_len), self.pad_token_id, dtype=torch.long
        )
        labels = torch.full(
            (n, max_len), -100, dtype=torch.long
        )
        attention_mask = torch.zeros(
            (n, max_len), dtype=torch.long
        )

        for i, item in enumerate(batch):
            length = len(item["input_ids"])
            input_ids[i, :length] = torch.tensor(item["input_ids"])
            labels[i, :length] = torch.tensor(item["labels"])
            attention_mask[i, :length] = 1

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            batch = {
                k: v.to(device, non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast("cuda", dtype=torch.float16):
                output = model(**batch)

            n_tokens = batch["labels"].ne(-100).sum().item()
            total_loss += output.loss.item() * n_tokens
            total_tokens += n_tokens

    model.train()
    return total_loss / max(total_tokens, 1)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", required=True)
    parser.add_argument("--val", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--lr", type=float, required=True)
    parser.add_argument("--dropout", type=float, required=True)
    parser.add_argument("--weight_decay", type=float, required=True)
    parser.add_argument("--epochs", type=int, default=1)
    parser.add_argument("--patience", type=int, default=3)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ["LOCAL_RANK"])

    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")

    torch.manual_seed(args.seed + rank)

    if rank == 0:
        os.makedirs(args.output, exist_ok=True)

    dist.barrier()

    train_rows = torch.load(
        args.train, map_location="cpu", weights_only=False
    )
    val_rows = torch.load(
        args.val, map_location="cpu", weights_only=False
    )

    train_dataset = EGFRDataset(train_rows)
    val_dataset = EGFRDataset(val_rows)

    tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen3.5-4B",
        padding_side="right"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    collator = CompletionOnlyCollator(tokenizer.pad_token_id)

    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=args.seed
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=1,
        sampler=train_sampler,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True
    )

    val_loader = None
    if rank == 0:
        val_loader = DataLoader(
            val_dataset,
            batch_size=1,
            shuffle=False,
            collate_fn=collator,
            num_workers=0,
            pin_memory=True
        )

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        "Qwen/Qwen3.5-4B",
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": local_rank},
        low_cpu_mem_usage=True
    )

    model.config.use_cache = False
    model.config.text_config.use_cache = False

    language_linear_modules = [
        name for name, module in model.named_modules()
        if isinstance(module, torch.nn.Linear)
        and name.startswith("model.language_model.")
    ]

    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    model = get_peft_model(
        model,
        LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=args.dropout,
            bias="none",
            target_modules=language_linear_modules,
            task_type="CAUSAL_LM"
        )
    )

    model = DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank,
        find_unused_parameters=False,
        broadcast_buffers=False,
        gradient_as_bucket_view=True
    )

    optimizer = bnb.optim.PagedAdamW8bit(
        [p for p in model.parameters() if p.requires_grad],
        lr=args.lr,
        betas=(0.9, 0.999),
        weight_decay=args.weight_decay
    )

    grad_accum = 4
    steps_per_epoch = math.ceil(len(train_loader) / grad_accum)
    total_steps = steps_per_epoch * args.epochs

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(total_steps * 0.05)),
        num_training_steps=total_steps
    )

    scaler = torch.amp.GradScaler("cuda")
    best_val = float("inf")
    patience_count = 0
    history = []

    for epoch in range(args.epochs):
        train_sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)

        total_train_loss = 0.0
        total_train_tokens = 0
        num_batches = len(train_loader)

        for step, batch in enumerate(train_loader):
            batch = {
                k: v.to(device, non_blocking=True)
                for k, v in batch.items()
            }

            group_start = (step // grad_accum) * grad_accum
            group_end = min(group_start + grad_accum, num_batches)
            group_size = group_end - group_start
            sync_step = (step + 1 == group_end)

            sync_context = nullcontext() if sync_step else model.no_sync()

            with sync_context:
                with torch.autocast("cuda", dtype=torch.float16):
                    output = model(**batch)
                    loss = output.loss

                n_tokens = batch["labels"].ne(-100).sum().item()
                total_train_loss += loss.detach().item() * n_tokens
                total_train_tokens += n_tokens

                scaler.scale(loss / group_size).backward()

            if sync_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    1.0
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            if rank == 0 and (step + 1) % 100 == 0:
                print(
                    f"Epoch {epoch+1}/{args.epochs} "
                    f"Batch {step+1}/{num_batches} "
                    f"Loss {loss.item():.4f}",
                    flush=True
                )

        stats = torch.tensor(
            [total_train_loss, total_train_tokens],
            dtype=torch.float64,
            device=device
        )
        dist.all_reduce(stats, op=dist.ReduceOp.SUM)
        train_loss = (stats[0] / stats[1]).item()

        dist.barrier()

        if rank == 0:
            val_loss = evaluate(model.module, val_loader, device)

            print(
                f"Epoch {epoch+1}: train={train_loss:.5f}, "
                f"validation={val_loss:.5f}",
                flush=True
            )

            history.append({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "validation_loss": val_loss
            })

            if val_loss < best_val:
                best_val = val_loss
                patience_count = 0

                if args.epochs > 1:
                    model.module.save_pretrained(
                        os.path.join(args.output, "best_adapter")
                    )
                    tokenizer.save_pretrained(
                        os.path.join(args.output, "best_adapter")
                    )
            else:
                patience_count += 1

            with open(os.path.join(args.output, "metrics.json"), "w") as f:
                json.dump({
                    "best_validation_loss": best_val,
                    "history": history
                }, f, indent=2)

        # Broadcast early-stopping decision to both processes.
        stop_tensor = torch.tensor(
            [1 if rank == 0 and patience_count >= args.patience else 0],
            device=device
        )
        dist.broadcast(stop_tensor, src=0)

        dist.barrier()

        if stop_tensor.item():
            if rank == 0:
                print("Early stopping triggered.", flush=True)
            break

        gc.collect()
        torch.cuda.empty_cache()

    # For one-epoch CV jobs, record the score without saving an adapter.
    dist.barrier()
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/egfr_ddp_worker.py


In [29]:
import os
import sys
import json
import torch
import subprocess
import optuna
import pandas as pd

from sklearn.model_selection import GroupKFold

WORKER = "/kaggle/working/egfr_ddp_worker.py"
TRIALS_DIR = f"{OUTPUT_DIR}/optuna_trials"
FINAL_DIR = f"{OUTPUT_DIR}/final_model"

os.makedirs(TRIALS_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

# Two GPUs, one process per GPU.
def launch_ddp(train_path, val_path, output_path,
               lr, dropout, weight_decay, epochs, patience=3):

    command = [
        sys.executable, "-m", "torch.distributed.run",
        "--standalone",
        "--nnodes=1",
        "--nproc_per_node=2",
        WORKER,
        "--train", train_path,
        "--val", val_path,
        "--output", output_path,
        "--lr", str(lr),
        "--dropout", str(dropout),
        "--weight_decay", str(weight_decay),
        "--epochs", str(epochs),
        "--patience", str(patience),
        "--seed", "42"
    ]

    env = os.environ.copy()
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["OMP_NUM_THREADS"] = "4"

    subprocess.run(command, check=True, env=env)


# Grouped three-fold CV on the training pool.
groups = train_pool["mutation_group"].to_numpy()
gkf = GroupKFold(n_splits=3)

folds = list(gkf.split(train_pool, groups=groups))

print("CV folds:", len(folds))
print("CV training examples:", len(train_pool))
print("Final holdout examples:", len(holdout))

def objective(trial):

    lr = trial.suggest_float(
        "learning_rate", 3e-5, 2e-4, log=True
    )
    dropout = trial.suggest_float(
        "lora_dropout", 0.0, 0.10
    )
    weight_decay = trial.suggest_float(
        "weight_decay", 0.0, 0.05
    )

    fold_scores = []

    print(f"\n========== TRIAL {trial.number} ==========")
    print({
        "learning_rate": lr,
        "lora_dropout": dropout,
        "weight_decay": weight_decay
    })

    for fold_number, (train_idx, val_idx) in enumerate(folds):

        fold_train = train_pool.iloc[train_idx]
        fold_val = train_pool.iloc[val_idx]

        # Verify mutation groups do not overlap.
        assert set(fold_train["mutation_group"]).isdisjoint(
            set(fold_val["mutation_group"])
        )

        fold_dir = f"{TRIALS_DIR}/trial_{trial.number}_fold_{fold_number}"
        os.makedirs(fold_dir, exist_ok=True)

        train_path = f"{fold_dir}/train.pt"
        val_path = f"{fold_dir}/val.pt"

        save_rows(fold_train, train_path)
        save_rows(fold_val, val_path)

        print(
            f"\nTrial {trial.number}, Fold {fold_number + 1}/3: "
            f"{len(fold_train)} train, {len(fold_val)} validation"
        )

        launch_ddp(
            train_path=train_path,
            val_path=val_path,
            output_path=fold_dir,
            lr=lr,
            dropout=dropout,
            weight_decay=weight_decay,
            epochs=1,
            patience=1
        )

        with open(f"{fold_dir}/metrics.json") as f:
            result = json.load(f)

        score = result["best_validation_loss"]
        fold_scores.append(score)

        print(f"Fold {fold_number + 1} validation loss: {score:.5f}")

        trial.set_user_attr(
            f"fold_{fold_number + 1}_loss", score
        )

    mean_score = sum(fold_scores) / len(fold_scores)

    print(f"\nTrial {trial.number} mean CV loss: {mean_score:.5f}")

    return mean_score


# Only two trials to preserve time for final training.
study = optuna.create_study(
    direction="minimize",
    study_name="egfr_qwen35_grouped_cv"
)

study.optimize(objective, n_trials=2)

print("\n========== OPTUNA RESULTS ==========")
print("Best CV loss:", study.best_value)
print("Best parameters:", study.best_params)

with open(f"{OUTPUT_DIR}/optuna_results.json", "w") as f:
    json.dump({
        "best_cv_loss": study.best_value,
        "best_parameters": study.best_params,
        "all_trials": [
            {
                "number": t.number,
                "value": t.value,
                "params": t.params,
                "fold_scores": t.user_attrs
            }
            for t in study.trials
        ]
    }, f, indent=2)


# ============================================================
# FINAL TRAINING: UP TO 10 EPOCHS
# ============================================================

best = study.best_params

print("\n========== FINAL MODEL TRAINING ==========")
print("Using best CV parameters:", best)
print("Maximum epochs: 10")
print("Early stopping patience: 3")

launch_ddp(
    train_path=f"{DATA_DIR}/final_train.pt",
    val_path=f"{DATA_DIR}/final_val.pt",
    output_path=FINAL_DIR,
    lr=best["learning_rate"],
    dropout=best["lora_dropout"],
    weight_decay=best["weight_decay"],
    epochs=10,
    patience=3
)

print("\n========== EXPERIMENT COMPLETE ==========")
print("Best adapter:", f"{FINAL_DIR}/best_adapter")
print("Final metrics:", f"{FINAL_DIR}/metrics.json")
print("Optuna results:", f"{OUTPUT_DIR}/optuna_results.json")

[I 2026-11-15 03:08:20,543] A new study created in memory with name: egfr_qwen35_grouped_cv


CV folds: 3
CV training examples: 945
Final holdout examples: 181

========== TRIAL 0 ==========
{'learning_rate': 5.818492004022956e-05, 'lora_dropout': 0.042373912691531956, 'weight_decay': 0.03444202619576727}

Trial 0, Fold 1/3: 633 train, 312 validation


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1115 03:08:33.777742424 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
Loading weights: 100%|██████████| 723/723 [00:11<00:00, 61.21it/s] 
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` i

Epoch 1/1 Batch 100/317 Loss 0.6068
Epoch 1/1 Batch 200/317 Loss 0.7376
Epoch 1/1 Batch 300/317 Loss 0.4481
Epoch 1: train=1.23211, validation=1.01413
Fold 1 validation loss: 1.01413

Trial 0, Fold 2/3: 629 train, 316 validation


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1115 03:17:34.018392212 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
Loading weights: 100%|██████████| 723/723 [00:11<00:00, 62.29it/s] 
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` i

Epoch 1/1 Batch 100/315 Loss 1.4145
Epoch 1/1 Batch 200/315 Loss 0.7557
Epoch 1/1 Batch 300/315 Loss 0.5725
Epoch 1: train=1.25826, validation=0.90139
Fold 2 validation loss: 0.90139

Trial 0, Fold 3/3: 628 train, 317 validation


... (11 outputs left out)

Fold 3 validation loss: 0.61613

Trial 1 mean CV loss: 0.69209

========== OPTUNA RESULTS ==========
Best CV loss: 0.6920922451148952
Best parameters: {'learning_rate': 0.00012492574844542607, 'lora_dropout': 0.08663112745550294, 'weight_decay': 0.04171375460156042}

========== FINAL MODEL TRAINING ==========
Using best CV parameters: {'learning_rate': 0.00012492574844542607, 'lora_dropout': 0.08663112745550294, 'weight_decay': 0.04171375460156042}
Maximum epochs: 10
Early stopping patience: 3


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1115 04:02:55.400779275 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
Loading weights: 100%|██████████| 723/723 [00:11<00:00, 60.27it/s] 
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` i

Epoch 1/10 Batch 100/473 Loss 1.4342
Epoch 1/10 Batch 200/473 Loss 0.5392
Epoch 1/10 Batch 300/473 Loss 1.5257
Epoch 1/10 Batch 400/473 Loss 0.2343
Epoch 1: train=0.99653, validation=0.81664


Epoch 2/10 Batch 100/473 Loss 0.0920
Epoch 2/10 Batch 200/473 Loss 0.0732
Epoch 2/10 Batch 300/473 Loss 0.7185
Epoch 2/10 Batch 400/473 Loss 0.0787
Epoch 2: train=0.21013, validation=0.63262
Epoch 3/10 Batch 100/473 Loss 0.0176
Epoch 3/10 Batch 200/473 Loss 0.0118
Epoch 3/10 Batch 300/473 Loss 0.0117
Epoch 3/10 Batch 400/473 Loss 0.0391
Epoch 3: train=0.09124, validation=0.59321
Epoch 4/10 Batch 100/473 Loss 0.0378
Epoch 4/10 Batch 200/473 Loss 0.0355
Epoch 4/10 Batch 300/473 Loss 0.0304
Epoch 4/10 Batch 400/473 Loss 0.0003
Epoch 4: train=0.04776, validation=0.59087
Epoch 5/10 Batch 100/473 Loss 0.0020
Epoch 5/10 Batch 200/473 Loss 0.0014
Epoch 5/10 Batch 300/473 Loss 0.0005
Epoch 5/10 Batch 400/473 Loss 0.0012
Epoch 5: train=0.02617, validation=0.63740
Epoch 6/10 Batch 100/473 Loss 0.0083
Epoch 6/10 Batch 200/473 Loss 0.0140
Epoch 6/10 Batch 300/473 Loss 0.0128
Epoch 6/10 Batch 400/473 Loss 0.0094
Epoch 6: train=0.01456, validation=0.66137
Epoch 7/10 Batch 100/473 Loss 0.0001
Epoch 7/

In [30]:
import json
import os

BASE = "/kaggle/working/egfr_qwen35_optuna_ddp"

with open(f"{BASE}/final_model/metrics.json") as f:
    final_metrics = json.load(f)

with open(f"{BASE}/optuna_results.json") as f:
    optuna_results = json.load(f)

print("BEST FINAL VALIDATION LOSS:")
print(final_metrics["best_validation_loss"])

print("\nFINAL TRAINING HISTORY:")
for row in final_metrics["history"]:
    print(row)

print("\nBEST OPTUNA PARAMETERS:")
print(optuna_results["best_parameters"])

print("\nBEST CROSS-VALIDATION LOSS:")
print(optuna_results["best_cv_loss"])

print("\nADAPTER FILES:")
print(os.listdir(f"{BASE}/final_model/best_adapter"))

BEST FINAL VALIDATION LOSS:
0.5908670665332176

FINAL TRAINING HISTORY:
{'epoch': 1, 'train_loss': 0.9965273355354582, 'validation_loss': 0.816643453079922}
{'epoch': 2, 'train_loss': 0.21012868179554056, 'validation_loss': 0.6326187069793502}
{'epoch': 3, 'train_loss': 0.09124281117796512, 'validation_loss': 0.593210071803407}
{'epoch': 4, 'train_loss': 0.04776475958982572, 'validation_loss': 0.5908670665332176}
{'epoch': 5, 'train_loss': 0.02616683051309233, 'validation_loss': 0.6373976096624685}
{'epoch': 6, 'train_loss': 0.014561000204498362, 'validation_loss': 0.6613704924129188}
{'epoch': 7, 'train_loss': 0.009683622997000407, 'validation_loss': 0.6686227031340439}

BEST OPTUNA PARAMETERS:
{'learning_rate': 0.00012492574844542607, 'lora_dropout': 0.08663112745550294, 'weight_decay': 0.04171375460156042}

BEST CROSS-VALIDATION LOSS:
0.6920922451148952

ADAPTER FILES:
['chat_template.jinja', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.jso

In [31]:
TEST_PATH = "/kaggle/input/datasets/rudrakshjoshi5/ic-egfr-dataset/EGFR_RESEARCH_DATASET/mutation_scientist/sft_test.csv"

In [32]:
import os
import json
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    Qwen3_5ForConditionalGeneration
)
from peft import PeftModel

MODEL_ID = "Qwen/Qwen3.5-4B"
ADAPTER_PATH = "/kaggle/working/egfr_qwen35_optuna_ddp/final_model/best_adapter"
TEST_PATH = "/kaggle/input/datasets/rudrakshjoshi5/ic-egfr-dataset/EGFR_RESEARCH_DATASET/mutation_scientist/sft_test.csv"

OUTPUT_DIR = "/kaggle/working/egfr_test_evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(ADAPTER_PATH)
assert os.path.exists(TEST_PATH)

test_df = pd.read_csv(TEST_PATH)
print("Test examples:", len(test_df))
print("Columns:", test_df.columns.tolist())

assert {"prompt", "completion"}.issubset(test_df.columns)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load base model
base_model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True
)

# Attach trained adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
model.config.use_cache = True

print("Model loaded successfully.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

Test examples: 215
Columns: ['prompt', 'completion', 'task_type', 'mutation_id', 'profile_id', 'compound_id', 'evidence_ids', 'source_ids', 'review_status', 'annotation_confidence']


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Model loaded successfully.
GPU memory allocated: 7.55 GB


In [33]:
def encode_example(prompt, completion):
    prompt_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": str(prompt)}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=False,
        enable_thinking=False
    )

    full_ids = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": str(prompt)},
            {"role": "assistant", "content": str(completion)}
        ],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False,
        enable_thinking=False
    )

    prompt_length = len(prompt_ids)

    if full_ids[:prompt_length] != prompt_ids:
        raise ValueError("Prompt/full conversation prefix mismatch.")

    full_ids = full_ids[:1024]
    labels = full_ids.copy()
    labels[:prompt_length] = [-100] * min(prompt_length, len(labels))

    if all(x == -100 for x in labels):
        return None

    return {
        "input_ids": torch.tensor(full_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }


losses = []
token_counts = []
failed_loss_rows = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test loss"):
    try:
        encoded = encode_example(row["prompt"], row["completion"])

        if encoded is None:
            failed_loss_rows.append(idx)
            continue

        input_ids = encoded["input_ids"].unsqueeze(0).to("cuda:0")
        labels = encoded["labels"].unsqueeze(0).to("cuda:0")
        attention_mask = torch.ones_like(input_ids)

        with torch.inference_mode():
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

        supervised_tokens = (labels != -100).sum().item()
        losses.append(float(outputs.loss.item()))
        token_counts.append(supervised_tokens)

        del input_ids, labels, attention_mask, outputs

    except Exception as e:
        failed_loss_rows.append(idx)
        print(f"Loss error at row {idx}: {e}")

test_loss = float(np.average(losses, weights=token_counts)) if losses else float("nan")

print("\nTEST LOSS RESULTS")
print("=================")
print("Evaluated examples:", len(losses))
print("Failed examples:", len(failed_loss_rows))
print("Token-weighted test loss:", round(test_loss, 5))
print("Mean per-example loss:", round(float(np.mean(losses)), 5))

Test loss:   0%|          | 0/215 [00:00<?, ?it/s]


TEST LOSS RESULTS
Evaluated examples: 215
Failed examples: 0
Token-weighted test loss: 0.19399
Mean per-example loss: 0.2259


In [34]:
MAX_NEW_TOKENS = 256

predictions = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating"):
    prompt = str(row["prompt"])

    try:
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            enable_thinking=False
        )

        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
        prompt_length = inputs["input_ids"].shape[1]

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        generated_ids = output_ids[0, prompt_length:]
        answer = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        predictions.append(answer)

    except Exception as e:
        print(f"Generation error at row {idx}: {e}")
        predictions.append("")

    if idx % 25 == 0:
        torch.cuda.empty_cache()

test_df["generated_completion"] = predictions

print("Generated:", len(predictions))
print("Empty predictions:", sum(not x for x in predictions))

Generating:   0%|          | 0/215 [00:00<?, ?it/s]

[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Generated: 215
Empty predictions: 0


In [35]:
prediction_path = os.path.join(OUTPUT_DIR, "test_predictions.csv")
test_df.to_csv(prediction_path, index=False)

print("Saved:", prediction_path)

# Inspect 10 examples
for i in range(min(10, len(test_df))):
    row = test_df.iloc[i]

    print("\n" + "=" * 90)
    print(f"Example {i+1}")
    print("Mutation:", row.get("mutation_id", "N/A"))
    print("Task:", row.get("task_type", "N/A"))
    print("\nPROMPT:\n", row["prompt"])
    print("\nREFERENCE:\n", row["completion"])
    print("\nGENERATED:\n", row["generated_completion"])

Saved: /kaggle/working/egfr_test_evaluation/test_predictions.csv

Example 1
Mutation: EGFR_L861Q
Task: mutation_mechanism

PROMPT:
 Explain the EGFR L861Q mutation: its type, location, functional classification and relevance to EGFR inhibitors. Wild-type/mutant residues: L->Q at position 861.

REFERENCE:
 EGFR L861Q is a missense in the Protein kinase domain (UniProt P00533 numbering). It is classified as activating. Exon-21 atypical activating mutation. Biochemical or cellular validation is required to quantify the magnitude of any sensitivity or resistance in a given context.

GENERATED:
 EGFR L861Q is a missense in the Protein kinase domain (UniProt P00533 numbering). It is classified as activating. Activation-loop activating mutation; TKI-sensitising. Biochemical or cellular validation is required to quantify the magnitude of any sensitivity or resistance in a given context.

Example 2
Mutation: EGFR_L861Q
Task: mutation_classification

PROMPT:
 Is EGFR L861Q an activating/sensitis

In [36]:
summary = {
    "model": MODEL_ID,
    "adapter": ADAPTER_PATH,
    "test_examples": len(test_df),
    "teacher_forced_token_weighted_loss": test_loss,
    "mean_per_example_loss": float(np.mean(losses)) if losses else None,
    "loss_evaluated_examples": len(losses),
    "loss_failed_examples": len(failed_loss_rows),
    "generation_failed_examples": sum(not x for x in predictions),
    "max_new_tokens": MAX_NEW_TOKENS,
    "decoding": "greedy"
}

with open(os.path.join(OUTPUT_DIR, "evaluation_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "model": "Qwen/Qwen3.5-4B",
  "adapter": "/kaggle/working/egfr_qwen35_optuna_ddp/final_model/best_adapter",
  "test_examples": 215,
  "teacher_forced_token_weighted_loss": 0.19398960740609128,
  "mean_per_example_loss": 0.2259015424179758,
  "loss_evaluated_examples": 215,
  "loss_failed_examples": 0,
  "generation_failed_examples": 0,
  "max_new_tokens": 256,
  "decoding": "greedy"
}


In [37]:
import os
import zipfile
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
ZIP_PATH = WORKING_DIR / "EGFR_MutationScientist_complete_backup.zip"

# Remove an older backup, if present
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# Calculate total source size
total_size = sum(
    f.stat().st_size
    for f in WORKING_DIR.rglob("*")
    if f.is_file() and f != ZIP_PATH
)

print(f"Files to archive: {sum(1 for f in WORKING_DIR.rglob('*') if f.is_file() and f != ZIP_PATH)}")
print(f"Uncompressed size: {total_size / (1024**3):.2f} GB")
print("Creating ZIP...")

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=4
) as zf:
    for file in WORKING_DIR.rglob("*"):
        if file.is_file() and file != ZIP_PATH:
            zf.write(file, file.relative_to(WORKING_DIR))

print("\nBACKUP COMPLETE")
print(f"Location: {ZIP_PATH}")
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**3):.2f} GB")

Files to archive: 41
Uncompressed size: 0.29 GB
Creating ZIP...

BACKUP COMPLETE
Location: /kaggle/working/EGFR_MutationScientist_complete_backup.zip
ZIP size: 0.23 GB
